# 🟡 Solution: FiLM Layer

### Exercise contract
Use an `nn.Module` with registered trainable conditioning parameters. Input x is `(B, S, D)`, condition is `(B, cond_dim)`, and output is `(B, S, D)` (D is the feature/hidden dimension). Each sample gets its own scale/shift vectors, shared across its tokens. Support autograd through x, condition, and the conditioning parameters.

Internal names are unrestricted: separate projections and a fused projection plus `chunk` are both valid. The judge exercises nonzero parameters and compares behavior; it does not require `gamma_fc`, `beta_fc`, `norm`, or `cond_mlp`.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class FiLM(nn.Module):
    def __init__(self, feature_dim: int, cond_dim: int):
        super().__init__()
        self.gamma_fc = nn.Linear(cond_dim, feature_dim)
        self.beta_fc = nn.Linear(cond_dim, feature_dim)
    
    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        gamma = self.gamma_fc(cond).unsqueeze(1)  # (batch, 1, feature_dim)
        beta = self.beta_fc(cond).unsqueeze(1)  # (batch, 1, feature_dim)
        return gamma * x + beta

In [ ]:
# Verify
film = FiLM(feature_dim=64, cond_dim=32)
x = torch.randn(2, 10, 64)
cond = torch.randn(2, 32)
out = film(x, cond)
print(f"Output shape: {out.shape}")

In [ ]:
from torch_judge import check
check("film")